# Early fusion

In [3]:
import pandas as pd
import numpy as np
import warnings
import gc
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_validate, GroupShuffleSplit, cross_val_predict
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

In [4]:
df = pd.read_pickle(r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl")

In [59]:
df.head()

,ModelID,SMILES,SequencingID,TSPAN6 (7105),SCYL3 (57147),BAD (572),LAP3 (51056),SNX11 (29916),CASP10 (843),CFLAR (8837),...,Bit_1022,Bit_1023,Donor,Acceptor,Aromatic,Hydrophobe,LumpedHydrophobe,PosIonizable,NegIonizable,ZnBinder
0,ACH-000001,B(C1=CC2=CC=CC=C2S1)(O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,2.0,2.0,2.0,3.0,2.0,0.0,0.0,0.0
1,ACH-000001,B(C1=CC=CC=C1)(C2=CC=CC=C2)OCCN,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,1.0,1.0,2.0,2.0,2.0,1.0,0.0,0.0
2,ACH-000001,C#CCCCCCCCCCCCCCCCC(=O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,1.0,2.0,0.0,16.0,0.0,0.0,1.0,1.0
3,ACH-000001,C(C(=O)O)S,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,2.0,2.0,0.0,1.0,0.0,0.0,1.0,2.0
4,ACH-000001,C(C(C(=O)O)N)SCC(=O)O,CDS-VqxBGH,5.490942,2.306714,5.959615,5.933639,4.679018,2.276571,4.200748,...,0,0,3.0,4.0,0.0,2.0,0.0,1.0,2.0,2.0


In [11]:
def early_fusion(df, target, model_name):
    if 'Bit_0' not in df.columns:
        fp_df = pd.DataFrame(df['MorganFP'].tolist(), index=df.index).astype('uint8')
        fp_df.columns = [f'Bit_{i}' for i in range(fp_df.shape[1])]
        df = pd.concat([df, fp_df], axis=1)
        
    if 'Donor' not in df.columns:
        expanded_features = pd.DataFrame(df['PharmacophoreFeatures'].tolist(), index=df.index).astype('float32')
        df = pd.concat([df.drop('PharmacophoreFeatures', axis=1), expanded_features], axis=1)
        df = df.fillna(0)
    # training feature set
    X = pd.concat([df.loc[:, ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']],
                        df.loc[:, df.columns.str.startswith('Bit_')],
                        df.filter(regex=r'.* \(.*\)')], axis=1)
    X_asList = X.values.astype('float32')
    y = df[target].values.astype('float32') # y: drug response
    # train-test split
    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=df['ModelID'].values))
    X_train, X_test = X_asList[train_idx], X_asList[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    groups = df.iloc[train_idx]['ModelID'].values # to ensure held-out validation

    if model_name == 'RandomForest':
        # initialize the Random Forest Regressor
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=15,
            min_samples_leaf=5,
            max_features='sqrt',
            n_jobs=-1,
            random_state=42
        )

        # perform Cross-Validation
        print("Starting Cross-Validation on Training Data...")
        cv_results = cross_validate(
            model, X_train, y_train, 
            groups=groups, 
            cv=GroupKFold(n_splits=5),
            scoring=['neg_mean_squared_error', 'r2'],
            return_train_score=True
        )

        # output Results
        mse_scores = -cv_results['test_neg_mean_squared_error']
        rmse_scores = np.sqrt(mse_scores)
        r2_scores = cv_results['test_r2']

        print(f"--- Multimodal (Random Forest - Early Fusion) Performance ---")
        print(f"R² Score: {np.mean(r2_scores):.4f}")
        print(f"RMSE:     {np.mean(rmse_scores):.4f}")
        print(f"------------------------------------")

        # fit the model on the training data
        model.fit(X_train, y_train)

        # test fit
        y_pred = model.predict(X_test)
        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        print(f"Test R² Score: {test_r2:.4f}")
        print(f"Test RMSE:     {test_rmse:.4f}")
        # feature importance
        all_features = X.columns
        importances = model.feature_importances_
        gene_cols = X.filter(regex=r'.* \(.*\)').columns
        features_df = pd.DataFrame({
            'Feature_Name': all_features,
            'Importance': importances,
            'Type': ['Genetic' if f in gene_cols else 'Molecular structure' for f in all_features]
        })
        # Filtere irrelevante Features heraus (Importance nahe 0)
        selected_features = features_df[features_df['Importance'] > 0.0001].copy()
        chem_selected = selected_features[selected_features['Type'] == 'Molecular structure']
        bio_selected = selected_features[selected_features['Type'] == 'Genetic']
        
        print(f"Total number of selected features: {len(selected_features)} von {len(all_features)}")
        print(f" -> Used chemical features:  {len(chem_selected)}")
        print(f" -> Used biological genes:    {len(bio_selected)}")
        
        print(f"\nTop 10 most important features:")
        print(features_df.sort_values(by='Importance', ascending=False).head(10))

    elif model_name == 'ElasticNet':
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=GroupKFold(n_splits=5).split(X_train, y_train, groups=groups),
                max_iter=8000,
                alphas=20,
                tol=1e-3,
                random_state=42,
                n_jobs=1
            ))
        ])

        pipeline.fit(X_train, y_train)

        train_r2_global = pipeline.score(X_train, y_train)
        fitted_model = pipeline.named_steps['model']
        best_alpha_idx = np.where(fitted_model.alphas_ == fitted_model.alpha_)[0][0]
        mean_mse_best_alpha = np.mean(fitted_model.mse_path_[best_alpha_idx])
        variance_y_train = np.var(y_train)
        train_cv_r2 = 1 - (mean_mse_best_alpha / variance_y_train)

        y_pred = pipeline.predict(X_test)
        test_r2 = pipeline.score(X_test, y_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        print("\n" + "="*40)
        print(f"--- Multimodal (Elastic Net - Early Fusion) Performance ---")
        print("="*40)
        print(f"Chosen Alpha:           {fitted_model.alpha_:.6f}")
        print(f"Chosen L1-Ratio:        {fitted_model.l1_ratio_:.2f}")
        print("-"*40)
        print(f"Global Training R² Score:  {train_r2_global:.4f}")
        print(f"Internal CV Trainings R²:     {train_cv_r2:.4f}")
        print(f"Held-Out Test R² Score:       {test_r2:.4f}")
        print("="*40)

        print(f"Test RMSE:     {test_rmse:.4f}")

        # analyze feature importance
        final_model = pipeline.named_steps['model']
        coefs = final_model.coef_

        # Create a summary table
        feature_names = X.columns
        features_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
        features_df['Abs_Coef'] = features_df['Coefficient'].abs()

        # Filter for features the model didn't set to zero
        selected_features = features_df[features_df['Coefficient'] != 0]

        print(f"\nElastic Net selected {len(selected_features)} features out of {len(feature_names)}.")
        print(f"Top 5 Positive Features (Increase {target}):")
        print(features_df.sort_values(by='Coefficient', ascending=False).head(5))
        print(f"\nTop 5 Negative Features (Decrease {target}):")
        print(features_df.sort_values(by='Coefficient', ascending=True).head(5))

## Early fusion with Random Forest

In [ ]:
early_fusion(df, target='LN_IC50', model_name='RandomForest')

Starting Cross-Validation on Training Data...
--- Multimodal (early fusion) Performance ---
R² Score: 0.5544
RMSE:     1.8868
------------------------------------
Test R² Score: 0.5546
Test RMSE:     1.8753
Total number of selected features: 921 von 2010
 -> Used chemical features:  651
 -> Used biological genes:    270

Top 10 most important features:
     Feature_Name  Importance                 Type
455       Bit_447    0.013441  Molecular structure
0           Donor    0.013330  Molecular structure
3      Hydrophobe    0.012498  Molecular structure
584       Bit_576    0.011212  Molecular structure
719       Bit_711    0.010133  Molecular structure
522       Bit_514    0.009569  Molecular structure
895       Bit_887    0.009402  Molecular structure
6    NegIonizable    0.008754  Molecular structure
641       Bit_633    0.008677  Molecular structure
432       Bit_424    0.008551  Molecular structure


In [ ]:
early_fusion(df, target='AUC', model_name='RandomForest')

Starting Cross-Validation on Training Data...
--- Multimodal (early fusion) Performance ---
R² Score: 0.4910
RMSE:     0.1067
------------------------------------
Test R² Score: 0.5104
Test RMSE:     0.1013
Total number of selected features: 1438 von 2010
 -> Used chemical features:  630
 -> Used biological genes:    808

Top 10 most important features:
    Feature_Name  Importance                 Type
522      Bit_514    0.019529  Molecular structure
584      Bit_576    0.015900  Molecular structure
573      Bit_565    0.014119  Molecular structure
484      Bit_476    0.012637  Molecular structure
577      Bit_569    0.012171  Molecular structure
719      Bit_711    0.011588  Molecular structure
367      Bit_359    0.011096  Molecular structure
410      Bit_402    0.010125  Molecular structure
7       ZnBinder    0.009546  Molecular structure
609      Bit_601    0.009128  Molecular structure


## Early fusion with Elastic Net

In [ ]:
early_fusion(df, target='LN_IC50', model_name='ElasticNet')

In [ ]:
early_fusion(df, target='AUC', model_name='ElasticNet')

# Late fusion

In late fusion, you would take the outputs (predictions or confidence scores) from these two separate models and combine them. This combination can be done in several ways:

    Averaging: Simple or weighted averaging of confidence scores.
    Voting: Each model "votes" for a class, and the majority wins.
    Product Rule: Multiplying probabilities (assuming independence).
    Small Model: Using another simple model (like a logistic regression or a small neural network) that takes the individual predictions as input and learns how to combine them.

I think it's gonna fail because my individual models are not performing well...

In [3]:
def late_fusion(df, target, model_name):
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    gss = GroupShuffleSplit(n_splits=1, train_size=0.8, test_size=0.2, random_state=42)
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
    X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
    y = df[target].values.astype('float32')

    
    train_idx, test_idx_drg = next(gss.split(df, groups=df['DRUG_ID']))
    train_idx_cl, test_idx_cl = next(gss.split(df, groups=df['ModelID']))
    df_train_drug = df.iloc[train_idx_drug]
    df_train_cl = df.iloc[train_idx_cl]
    pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
    X_train_chem = pd.concat([df_train_drug.loc[:, pharmacophores], df_train_drug.loc[:, df_train_drug.columns.str.startswith('Bit_')]], axis=1).values.astype('float32')
    X_train_gene = df_train_cl.filter(regex=r'.* \(.*\)').values.astype('float32')
    y_train_chem = df_train_drug[target].values.astype('float32')
    y_train_gene = df_train_cl[target].values.astype('float32')
    groups_chem = df_train_drug['DRUG_ID'].values
    groups_gene = df_train_cl['ModelID'].values

    # Extract Test Matrices (Full size, no downsampling)
    X_test_chem = pd.concat([df.loc[df.index[test_idx_drug], pharmacophores], df.loc[df.index[test_idx_drug], df.columns.str.startswith('Bit_')]], axis=1).values.astype('float32')
    y_test_chem = df.loc[df.index[test_idx_drug], target].values.astype('float32')
    X_test_gene = df.loc[df.index[test_idx_cl], df.filter(regex=r'.* \(.*\)').columns].values.astype('float32')
    y_test_gene = df.loc[df.index[test_idx_cl], target].values.astype('float32')

    del df_train_drug, df_train_cl


    gc.collect()
    # single-modal models
    if model_name == 'ElasticNet':
        print("Elastic Net - Late Fusion")
        pipeline_chem = Pipeline([
            ('selector', VarianceThreshold(threshold=0.01)),
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=5,
                random_state=42,
                max_iter=7000,
                alphas=20,
                tol=1e-3,
                n_jobs=-1
                ))
        ])
        pipeline_genomic = Pipeline([
            ('selector', VarianceThreshold(threshold=0.01)),
            ('scaler', StandardScaler()),
            ('model', ElasticNetCV(
                l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
                cv=5,
                random_state=42,
                max_iter=7000,
                alphas=20,
                tol=1e-3,
                n_jobs=-1
                ))
        ])
        
        print("Starting Cross-Validation for Genomic Model...")
        cv_genomic = cross_val_predict(
            pipeline_genomic, X_genomic, y,
            groups=df['ModelID'], cv=GroupKFold(n_splits=5)
        )
        print("Starting Cross-Validation for Chemical Model...")
        cv_chem = cross_val_predict(
            pipeline_chem, X_chem, y,
            groups=df['DRUG_ID'], cv=GroupKFold(n_splits=5)
        )

        print("LinearRegression...")
        # combine predictions (late fusion)
        # use linear regression for final prediction
        X_combined = np.column_stack([cv_genomic, cv_chem])
        lr = LinearRegression()
        lr.fit(X_combined, y)

        print("Fitting Chemical Model on Full Training Data...")
        pipeline_genomic.fit(X_genomic, y)
        print("Fitting Genomic Model on Full Training Data...")
        pipeline_chem.fit(X_chem, y)

        # using a combined split for final evaluation to ensure no data leakage
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(X_combined, y, groups=df['DRUG_ID']))
        X_test_gene = X_genomic[test_idx]
        X_test_chem = X_chem[test_idx]
        y_test = y[test_idx]

        y_pred_gen = pipeline_genomic.predict(X_test_gene)
        y_pred_chem = pipeline_chem.predict(X_test_chem)
        y_pred_meta = lr.predict(np.column_stack([y_pred_gen, y_pred_chem]))

        print(f"R² Genomic single-modality model: {r2_score(y_test, y_pred_gen):.4f}")
        print(f"R² Chem single-modality model:    {r2_score(y_test, y_pred_chem):.4f}")
        print(f"R² Late Fusion (meta model): {r2_score(y_test, y_pred_meta):.4f}")
        print("="*40)
        
         # Extract Decision-Learner Weights
        weight_chem, weight_gene = lr.coef_
        print("Decision-Learner Weights:")
        print(f" -> Weight for Chemical Model: {weight_chem:.4f}")
        print(f" -> Weight for Genomic Model:  {weight_gene:.4f}")

    elif model_name == 'RandomForest':
        print("Random Forest - Late Fusion")
        rf_genomic = RandomForestRegressor(n_estimators=80,max_depth=15,min_samples_leaf=5,n_jobs=4,max_features='sqrt', random_state=42)
        rf_chem = RandomForestRegressor(n_estimators=80,max_depth=15,min_samples_leaf=5,n_jobs=4,max_features='sqrt', random_state=42)
        print("Starting Cross-Validation for Genomic Model...")
        cv_genomic = cross_val_predict(
            rf_genomic, X_genomic, y, 
            groups=df['ModelID'], cv=GroupKFold(n_splits=5)
        )
        print("Starting Cross-Validation for Chemical Model...")
        cv_chem = cross_val_predict(
            rf_chem, X_chem, y, 
            groups=df['DRUG_ID'], cv=GroupKFold(n_splits=5)
        )
        
        print("LinearRegression...")
        X_combined = np.column_stack((cv_genomic, cv_chem))
        # use linear regression for final prediction
        lr = LinearRegression()
        lr.fit(X_combined, y)
        # train baseline models
        print("Fitting Genomic and Chemical Models on Full Training Data...")
        rf_genomic.fit(X_genomic, y)
        rf_chem.fit(X_chem, y)

        # testing
        # using a combined split for final evaluation to ensure no data leakage
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, test_idx = next(gss.split(X_combined, y, groups=df['DRUG_ID']))
        X_test_gene = X_genomic[test_idx]
        X_test_chem = X_chem[test_idx]
        y_test = y[test_idx]
        print("Evaluating on Test Set...")
        y_pred_gen = rf_genomic.predict(X_test_gene)
        y_pred_chem = rf_chem.predict(X_test_chem)
        y_pred_meta = lr.predict(np.column_stack([y_pred_gen, y_pred_chem]))
        
        print("="*40)
        print(f"R² Genomic single-modality model: {r2_score(y_test, y_pred_gen):.4f}")
        print(f"R² Chem single-modality model:    {r2_score(y_test, y_pred_chem):.4f}")
        print(f"R² Late Fusion (meta model): {r2_score(y_test, y_pred_meta):.4f}")
        print("="*40)
        
         # Extract Decision-Learner Weights
        weight_chem, weight_gene = lr.coef_
        print("Decision-Learner Weights:")
        print(f" -> Weight for Chemical Model: {weight_chem:.4f}")
        print(f" -> Weight for Genomic Model:  {weight_gene:.4f}")

In [72]:
late_fusion(df, target='LN_IC50', model_name='RandomForest')

Random Forest - Late Fusion
Starting Cross-Validation for Genomic Model...
Starting Cross-Validation for Chemical Model...
LinearRegression...
Fitting Genomic and Chemical Models on Full Training Data...
Evaluating on Test Set...
R² Genomic single-modality model: 0.1311
R² Chem single-modality model:    0.5567
R² Late Fusion (meta model): 0.4527
Decision-Learner Weights:
 -> Weight for Chemical Model: 1.1361
 -> Weight for Genomic Model:  0.4269


In [68]:
late_fusion(df, target='LN_IC50', model_name='ElasticNet')

Elastic Net - Late Fusion
Starting Cross-Validation for Genomic Model...
Starting Cross-Validation for Chemical Model...
LinearRegression...
Fitting Chemical Model on Full Training Data...
Fitting Genomic Model on Full Training Data...
R² Genomic single-modality model: 0.0887
R² Chem single-modality model:    0.6835
R² Late Fusion (meta model): 0.2356
Decision-Learner Weights:
 -> Weight for Chemical Model: 1.0246
 -> Weight for Genomic Model:  0.1137


In [5]:
# training data
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem = df[pharmacophores + list(df.columns[df.columns.str.startswith('Bit_')])].values.astype('float32')
y = df[target].values.astype('float32')

pipeline_chem = Pipeline([
    ('selector', VarianceThreshold(threshold=0.01)),
    ('scaler', StandardScaler()),
    ('model', ElasticNetCV(
        l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
        cv=5,
        random_state=42,
        max_iter=7000,
        alphas=20,
        tol=1e-3,
        n_jobs=4
        ))
])
pipeline_genomic = Pipeline([
    ('selector', VarianceThreshold(threshold=0.01)),
    ('scaler', StandardScaler()),
    ('model', ElasticNetCV(
        l1_ratio=[0.001, 0.01, 0.05, 0.1, 0.5, 0.7, 0.9, 0.99, 1], # model will find the best mix
        cv=5,
        random_state=42,
        max_iter=7000,
        alphas=20,
        tol=1e-3,
        n_jobs=4
        ))
])
# OOF predictions
cv_genomic = cross_val_predict(pipeline_genomic, X_genomic, y, groups=df['ModelID'], cv=GroupKFold(5))
cv_chem = cross_val_predict(pipeline_chem, X_chem, y, groups=df['DRUG_ID'], cv=GroupKFold(5))


In [6]:
lr = LinearRegression()

# split for a clean meta-model evaluation
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['ModelID']))

X_combined_train = np.column_stack([cv_genomic[train_idx], cv_chem[train_idx]])
lr.fit(X_combined_train, y[train_idx])

y_pred_meta = lr.predict(np.column_stack([cv_genomic[test_idx], cv_chem[test_idx]]))

In [7]:
r2 = r2_score(y[test_idx], y_pred_meta)
rmse = np.sqrt(mean_squared_error(y[test_idx], y_pred_meta))

print("\n" + "="*40)
print("     FINAL TEST-RESULTS (LATE FUSION)")
print("="*40)
print(f"R² Score: {r2:.4f}")
print(f"RMSE:     {rmse:.4f}")
print("="*40)


     FINAL TEST-RESULTS (LATE FUSION)
R² Score: 0.0523
RMSE:     2.7354


Trying another prediction ensembler

In [14]:
# fit final models on full data for feature importance analysis
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_, test_idx_ = next(gss.split(df, groups=df['ModelID']))


pipeline_genomic.fit(X_genomic[train_idx_], y[train_idx_])
pipeline_chem.fit(X_chem[train_idx_], y[train_idx_])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('selector', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"threshold threshold: float, default=0Features with a training-set variance lower than this threshold willbe removed. The default is to keep all features with non-zero variance,i.e. remove the features that have the same value in all samples.",0.01
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"l1_ratio l1_ratio: float or list of float, default=0.5Float between 0 and 1 passed to ElasticNet (scaling betweenl1 and l2 penalties). For ``l1_ratio = 0``the penalty is an L2 penalty. For ``l1_ratio = 1`` it is an L1 penalty.For ``0 < l1_ratio < 1``, the penalty is a combination of L1 and L2This parameter can be a list, in which case the differentvalues are tested by cross-validation and the one giving the bestprediction score is used. Note that a good choice of list ofvalues for l1_ratio is often to put more values close to 1(i.e. Lasso) and less close to 0 (i.e. Ridge), as in ``[.1, .5, .7,.9, .95, .99, 1]``.","[0.001, 0.01, ...]"
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path, used for each l1_ratio... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'


In [15]:
y_pred_gen = pipeline_genomic.predict(X_genomic[test_idx])
y_pred_chem = pipeline_chem.predict(X_chem[test_idx])

y_pred_meta_avg = (y_pred_gen + y_pred_chem) / 2

print("--- Simple Averaging ---")
print(f"R² Genomic: {r2_score(y[test_idx], y_pred_gen):.4f}")
print(f"R² Chem:    {r2_score(y[test_idx], y_pred_chem):.4f}")
print(f"R² Average: {r2_score(y[test_idx], y_pred_meta_avg):.4f}")
print(f"RMSE Average: {np.sqrt(mean_squared_error(y[test_idx], y_pred_meta_avg)):.4f}")

--- Simple Averaging ---
R² Genomic: 0.0460
R² Chem:    0.7270
R² Average: 0.5801
RMSE Average: 1.8207
